# Training Lora From Corpus Version 4 (Big)
This is stage 2 qa corpus training on top of v1

# Load model with Unsloth patching

In [1]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-8B",
    max_seq_length=1024,
    load_in_4bit=True,
)

print("Loaded model in 4-bit ✅")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-23 20:41:25 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.3: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.10.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+f204359.d20251014. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Loaded model in 4-bit ✅


# Apply LoRa adapter

In [2]:
peft_model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.2,
    target_modules=["q_proj","v_proj","o_proj","k_proj"],
    bias="none",
    use_gradient_checkpointing=True,
)

tokenizer.bos_token = None
tokenizer.eos_token = "</s>"  # Qwen3’s typical EOS token
tokenizer.pad_token = tokenizer.eos_token  # Common practice
model.config.bos_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
peft_model.config.bos_token_id = tokenizer.bos_token_id
peft_model.config.eos_token_id = tokenizer.eos_token_id
peft_model.config.pad_token_id = tokenizer.pad_token_id

print("Loaded peft model ✅")


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.2.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.10.3 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Loaded peft model ✅


# Load the dataset from corpus

In [4]:
import os 
import json

CORPUS_QA_DIR = "/storage/corpus/corpus_farsight"
BLOCK_SIZE = 1024  # max tokens per chunk
EOS_STR = "</s>"

tok = tokenizer 

# Ensure EOS/PAD exist and are consistent
_added = False
if tok.eos_token is None:
    tok.add_special_tokens({"eos_token": EOS_STR})
    _added = True
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    _added = True
if _added:
    model.resize_token_embeddings(len(tok))

# -----------------------
# 1) Load QA JSON files (array JSON or JSONL), normalize to strings
# -----------------------
def _normalize_one(q: str, a: str) -> str:
    q = (q or "").strip()
    a = (a or "").strip()
    # If answer already has </s>, remove so we enforce exactly one.
    if a.endswith(EOS_STR):
        a = a[: -len(EOS_STR)].rstrip()
    return f"Q: {q}\nA: {a}{EOS_STR}"

def load_qa_corpus(directory):
    texts = []
    for filename in os.listdir(directory):
        if not (filename.endswith(".json") or filename.endswith(".jsonl")):
            continue
        path = os.path.join(directory, filename)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            head = f.read(1)
            f.seek(0)
            if head == "[":  # JSON array
                try:
                    arr = json.load(f)
                    for obj in arr:
                        if isinstance(obj, dict) and "question" in obj and "answer" in obj:
                            texts.append(_normalize_one(obj["question"], obj["answer"]))
                except json.JSONDecodeError:
                    # fall back to line-by-line if malformed
                    f.seek(0)
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        try:
                            obj = json.loads(line)
                            if "question" in obj and "answer" in obj:
                                texts.append(_normalize_one(obj["question"], obj["answer"]))
                        except json.JSONDecodeError:
                            continue
            else:  # JSONL
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                        if "question" in obj and "answer" in obj:
                            texts.append(_normalize_one(obj["question"], obj["answer"]))
                    except json.JSONDecodeError:
                        continue
    return texts

raw_texts = load_qa_corpus(CORPUS_QA_DIR)
print(f"Loaded {len(raw_texts)} QA items from {CORPUS_QA_DIR} ✅")

Loaded 0 QA items from /storage/corpus/corpus_farsight ✅


# Tokenize with EOS appended (token id, not string)
We’ll build one long stream of ids and then pack into BLOCK_SIZE chunks.

In [4]:
from datasets import Dataset

def tokenize_append_eos(texts):
    # We already embedded one EOS in every record; we still emulate your flow and
    # append an EOS id after each example (harmless because Qwen treats </s> as EOS).
    enc = tok(texts, add_special_tokens=False)
    flat_ids = []
    append_id = tok.eos_token_id
    for ids in enc["input_ids"]:
        flat_ids.extend(ids)
        if append_id is not None:
            flat_ids.append(append_id)
    return flat_ids

flat_ids = tokenize_append_eos(raw_texts)

# -----------------------
# 2) Pack into fixed-length blocks
# -----------------------
def pack_ids_to_blocks(ids, block_size):
    blocks = []
    total = len(ids)
    usable = total - (total % block_size)
    ids = ids[:usable]
    for i in range(0, usable, block_size):
        chunk = ids[i : i + block_size]
        example = {
            "input_ids": chunk,
            "attention_mask": [1] * block_size,
            # Labels = input_ids for standard causal LM training
            "labels": chunk[:],
        }
        blocks.append(example)
    return Dataset.from_list(blocks)

train_dataset = pack_ids_to_blocks(flat_ids, BLOCK_SIZE)
print(f"Prepared {len(train_dataset)} packed training chunks of {BLOCK_SIZE} tokens ✅")


Prepared 2738 packed training chunks of 1024 tokens ✅


# Init Trainer Params

In [5]:
import os
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

# 1) Absolute, writable, persistent output dir
OUTPUT_DIR = "/storage/models/wtk-qwen3-beta-slim-lora-v4"

# 2) Build explicit TrainingArguments (NO dict here)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,      # reuse dir safely
    resume_from_checkpoint=True,    # set False if you want a totally clean start
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    lr_scheduler_type="cosine",
    learning_rate=1e-5,
    warmup_ratio=0.03, 
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_steps=100,  # Save checkpoint every 100 steps
    report_to="none",
    remove_unused_columns=False,
    metric_for_best_model="loss",
    greater_is_better=False,
    max_grad_norm=1.0,
    dataloader_num_workers=0,       # more stable on mounted storage
)

# 3) Build the trainer
#trainer = SFTTrainer(
#    model=peft_model,
#    tokenizer=tok,
#    train_dataset=train_dataset,
#    max_seq_length=BLOCK_SIZE,
#    args=training_args,
#)

# Our dataset already has labels; a simple collator is fine
data_collator = DataCollatorWithPadding(tokenizer=tok, padding="longest")

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=tok,
)

print(f"Created SFTTrainer ✅")


Created SFTTrainer ✅


/tmp/ipykernel_361/1016106345.py:43: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Train using SFTTrainer (new)

In [6]:
# 4) Start training (this might take a while)
trainer.train()
print("Training complete ✅")

# 5) Save trained model to storage
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

print("Training results saved ✅")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128247, 'bos_token_id': None, 'pad_token_id': 128247}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,738 | Num Epochs = 1 | Total steps = 172
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 19,922,944 of 32,782,046,208 (0.06% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.747700
20,1.761700
30,1.751500
40,1.722500
50,1.719400
60,1.710900
70,1.685700
80,1.671000
90,1.659300
100,1.641400


Training complete ✅
Training results saved ✅


# Resume training using SFTTrainer (only use if resuming)

In [ ]:
CHECKPOINT_DIR = OUTPUT_DIR + "/" + "checkpoint-500"

#4) Start training (this might take a while)
print(f"Resuming from checkpoint: {CHECKPOINT_DIR}")
trainer.train(resume_from_checkpoint=CHECKPOINT_DIR)
print("Training complete ✅")

# 5) Save trained model to storage
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

print("Training results saved ✅")

Resuming from checkpoint: /workspace/wtk-qwen3-beta-slim-lora-v3/checkpoint-500


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128247, 'bos_token_id': None, 'pad_token_id': 128247}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 31,156 | Num Epochs = 1 | Total steps = 1,948
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 26,738,688 of 30,558,861,312 (0.09% trained)
	save_steps: 100 (from args) != 500 (from trainer_state.json)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the envir

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


# Push LoRa to Huggingface

In [7]:
from huggingface_hub import HfApi, upload_folder

repo_id = "peers-ai/wtk-qwen3-beta-slim-lora-v4-B"
folder = OUTPUT_DIR  # contains adapter_config.json & adapter_model.bin

api = HfApi()
# create the repo if it doesn't exist
api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)

# upload all files in the folder
upload_folder(
    repo_id=repo_id,
    folder_path=folder,
    repo_type="model",
)
print(f"✅ Uploaded to https://huggingface.co/{repo_id}")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Uploaded to https://huggingface.co/peers-ai/wtk-qwen3-beta-slim-lora-v4-B
